# Samian ware: potter activity at production centres

This local Jupyter notebook joins two SPARQL queries from the NFDI4Objects Knowledge Graph — one on **potter–centre assignments** with fuzzy confidence scores, the other on **production-centre geographies** — into a single proportional-symbol map that makes the productive mass of Samian-ware workshops visible at a glance.

A companion browser-executable notebook, `n4okg-samian-potter-activity-live.qmd`, runs the same pipeline via Pyodide and Leaflet directly in the browser for readers who prefer a server-less environment.


## Requirements

```bash
pip install SPARQLWrapper pandas matplotlib folium
```


## About this notebook

Samian-ware potters are known from stamps, decorative signatures, and stylistic attribution. Assigning a potter to a production centre is often a judgement call — a stamp may point to one workshop, stylistic analysis to another, and sometimes a potter is recorded as active at *several* sites. The NFDI4Objects Knowledge Graph captures this uncertainty explicitly through the **Academic Meta Tool (AMT)** vocabulary: every potter–centre assignment carries an `amt:weight`, a value in `[0, 1]` expressing confidence — `1.0` for secure attributions, fractional values (often `0.5`) for vague assignments derived from rules such as "worked at kilnsite A *or* kilnsite B".

Summing those weights per centre yields a **productive-mass score** for each workshop: not a potter count, but a confidence-weighted estimate of how much potter activity was concentrated there. This notebook computes that score, joins it to the centre geographies, and plots the result.

### Why this dataset?

Fuzzy-score reasoning is a thread that runs through NFDI4Objects work on archaeological inference. The Samian-ware dataset is an unusually clean example: the same `amt:weight` mechanism feeds the AMT Python port's reasoning engine, and the scores here are the concrete output of that engine applied to real potter records. Visualising the weights spatially closes the loop between inference and interpretation.

### What you'll learn

- how to join two SPARQL result sets in pandas via a shared IRI column
- how `SUM(xsd:decimal(?w))` in SPARQL gives a per-group confidence aggregate
- how to build a proportional-symbol map with range-labelled legend, avoiding the common pitfall of scaling *radius* directly (which exaggerates large values)

### Data-context notes

- query 1 (`ProductionCentre + KilnRegion + geometry`) has one row per centre, around 100 in total
- query 3 (`Potter × Centre, weighted`) has many thousand rows — one per (potter, centre) pair, aggregated via `GROUP BY`
- scores typically cluster around a few discrete values: `1.0` for secure, `0.5` for two-way vague, `0.33…` for three-way, etc.
- centres may carry more than one kiln-region label; we keep the first encountered, matching query-1 behaviour
- a centre with no potter records has score 0 and is not plotted (inner-join semantics)

### Tooling notes

Locally we use `SPARQLWrapper`, `pandas`, `matplotlib` and `folium`. Radius is scaled as √score (area ∝ score) so that a centre with twice the score looks twice as busy, not four times as busy — the standard correction for proportional-symbol maps.


## Step 1 — Define the SPARQL queries

Two queries this time. The potter-activity query groups by centre and sums the confidence weights; the centre-geography query is the same one used in the production-centres notebook, reused here so the two notebooks can stand alone.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import math

USER_AGENT = "OER-Quarto-Notebook/1.0 (https://n4o-rse.github.io/oer-ta2-koeln/)"
SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"

QUERY_ACTIVITY = """
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX lado: <http://archaeology.link/ontology#>
PREFIX amt:  <http://academic-meta-tool.xyz/vocab#>
SELECT ?item ?itemLabel ?actorEntity ?actorEntityLabel (SUM(xsd:decimal(?w)) AS ?score)
WHERE {
  GRAPH <https://graph.nfdi4objects.net/collection/8> {
    ?actorEntity rdf:type lado:ActorEntity ;
                 lado:hasAMT ?stmt .
    ?stmt rdf:predicate lado:worksAtPlace ;
          rdf:object ?item ;
          amt:weight ?w .
    ?item rdf:type lado:ProductionCentre .
    OPTIONAL { ?item rdfs:label ?itemLabel . }
    OPTIONAL { ?actorEntity rdfs:label ?actorEntityLabel . }
  }
}
GROUP BY ?item ?itemLabel ?actorEntity ?actorEntityLabel
ORDER BY ?itemLabel DESC(?score)
"""

QUERY_CENTRES = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX lado: <http://archaeology.link/ontology#>
SELECT ?item ?itemLabel ?geo ?kilnregion WHERE {
  GRAPH <https://graph.nfdi4objects.net/collection/8> {
    ?item <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://archaeology.link/ontology#ProductionCentre> .
    ?item <http://www.opengis.net/ont/geosparql#hasGeometry> ?item_geom .
    ?item_geom <http://www.opengis.net/ont/geosparql#asWKT> ?geo .
    ?item rdfs:label ?itemLabel .
    ?item lado:clusteredAs ?kr .
    ?kr rdfs:label ?kilnregion .
  }
}
"""

def run_sparql(query):
    sp = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
    sp.setQuery(query)
    sp.setReturnFormat(JSON)
    return sp.query().convert()["results"]["bindings"]
print("✓ Helpers defined. Run the next cell to load the data.")


## Step 2 — Load the data

Three dataframes are assembled here: `df_assignments` with one row per potter–centre pair, `df_centres` with one row per centre (geo + region), and `df_activity` — the per-centre aggregate computed from `df_assignments` and joined to `df_centres`. Keeping the intermediate dataframes around makes the Step-4 exploration cell more flexible.


In [ ]:
import re

_WKT_POINT_RE = re.compile(r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)",
                           re.IGNORECASE)

def parse_wkt_point(wkt):
    """Parse 'POINT(lon lat)' (any case) into (lat, lon) ready for mapping libraries."""
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)


In [ ]:
# --- query 1: potter × centre with fuzzy score
bindings_a = run_sparql(QUERY_ACTIVITY)
df_assignments = pd.DataFrame([
    {
        "item": b["item"]["value"],
        "itemLabel": b.get("itemLabel", {}).get("value") or b["item"]["value"].rsplit("/", 1)[-1],
        "actorEntity": b["actorEntity"]["value"],
        "actorEntityLabel": b.get("actorEntityLabel", {}).get("value") or b["actorEntity"]["value"].rsplit("/", 1)[-1],
        "score": float(b["score"]["value"]),
    }
    for b in bindings_a
])

# --- query 2: centre → geo + kiln region
bindings_c = run_sparql(QUERY_CENTRES)
rows_c = []
for b in bindings_c:
    lat, lon = parse_wkt_point(b.get("geo", {}).get("value"))
    rows_c.append({
        "item": b["item"]["value"],
        "itemLabel": b["itemLabel"]["value"],
        "kilnregion": b["kilnregion"]["value"],
        "latitude": lat,
        "longitude": lon,
    })
df_centres = pd.DataFrame(rows_c)
df_centres = df_centres.dropna(subset=["latitude", "longitude"])
df_centres = df_centres.drop_duplicates(subset="item", keep="first").reset_index(drop=True)

# --- aggregate potter activity per centre
df_agg = df_assignments.groupby("item").agg(
    total_score=("score", "sum"),
    n_potters=("actorEntity", "nunique"),
).reset_index()

# --- join to geography
df_activity = df_centres.merge(df_agg, on="item", how="inner")
df_activity = df_activity.sort_values("total_score", ascending=False).reset_index(drop=True)

print(f"Assignments : {len(df_assignments):>6} rows")
print(f"Centres     : {len(df_centres):>6} rows")
print(f"Active centres (with ≥1 potter) : {len(df_activity)}")
df_activity.head()


## Step 3a — Top 15 centres by potter activity (sanity check)

A horizontal bar chart of the 15 most productive centres orders the map's visual ranking numerically. Bars are coloured by kiln region to match the map palette below, so the two views can be read as one.


In [ ]:
import matplotlib.pyplot as plt

PALETTE = [
    "#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#469990", "#9A6324", "#800000",
    "#808000", "#000075", "#e91e63", "#607d8b", "#795548",
]
regions = sorted(df_activity["kilnregion"].unique())
region_colors = {r: PALETTE[i % len(PALETTE)] for i, r in enumerate(regions)}

top15 = df_activity.head(15).iloc[::-1]
colors = [region_colors[r] for r in top15["kilnregion"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15["itemLabel"], top15["total_score"], color=colors)
ax.set_xlabel("Total potter activity (sum of AMT weights)")
ax.set_title("Top 15 Samian-ware production centres by potter activity")
for i, (v, n) in enumerate(zip(top15["total_score"], top15["n_potters"])):
    ax.text(v + max(top15["total_score"]) * 0.01, i,
            f"{v:.1f}  ({int(n)} potters)", va="center", fontsize=8)
plt.tight_layout()
plt.show()


## Step 3b — Proportional-symbol map

Each active centre is drawn as a circle whose **area** (not radius) scales with the total potter-activity score. Colour encodes kiln region, so the map simultaneously answers *where is the activity concentrated?* and *which regional tradition is each centre part of?*. A separate legend box in the bottom-left corner reports three size ranges; the overlay control embeds colour swatches for each region.


In [ ]:
import folium
from folium import Element
import math

score_max = float(df_activity["total_score"].max())
BUCKETS = [
    {"label": "1–10",  "radius_px": 6},
    {"label": "11–50", "radius_px": 12},
    {"label": "51+",   "radius_px": 22},
]

def radius_for(score, s_max=score_max):
    if s_max <= 0:
        return 5
    return 5 + 25 * math.sqrt(score / s_max)

top_potters_per_item = {}
for item, grp in df_assignments.groupby("item"):
    top = (grp.sort_values("score", ascending=False)
               .head(10)[["actorEntityLabel", "score"]]
               .to_dict("records"))
    top_potters_per_item[item] = top

lat_c = float(df_activity["latitude"].mean())
lon_c = float(df_activity["longitude"].mean())

m = folium.Map(location=[lat_c, lon_c], zoom_start=4, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    "https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    name="OpenStreetMap.HOT", attr="© OpenStreetMap contributors, HOT style"
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery", attr="Tiles © Esri — Source: Esri, Maxar, Earthstar Geographics"
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldTerrain", attr="Tiles © Esri — Source: USGS, Esri, TANA, DeLorme, NPS", max_zoom=13
).add_to(m)

region_groups = {r: folium.FeatureGroup(name=r).add_to(m) for r in regions}

for _, row in df_activity.iterrows():
    top = top_potters_per_item.get(row["item"], [])
    top_html = ""
    if top:
        top_html = '<ol style="margin:4px 0 0 18px;padding:0;font-size:12px">' \
            + "".join(
                f'<li>{p["actorEntityLabel"]} '
                f'<span style="color:#666">({p["score"]:.2f})</span></li>'
                for p in top
            ) + "</ol>"
    popup_html = (
        f"<b>{row['itemLabel']}</b><br>"
        f"<i>{row['kilnregion']}</i><br>"
        f"Total score: <b>{row['total_score']:.2f}</b><br>"
        f"Potters: <b>{int(row['n_potters'])}</b><br>"
        f'<span style="font-size:12px">Top potters by score:</span>'
        f"{top_html}"
        f'<a href="{row["item"]}" target="_blank" style="font-size:12px">View centre in N4O KG ↗</a>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=radius_for(float(row["total_score"])),
        color=region_colors[row["kilnregion"]],
        weight=1,
        fill=True,
        fill_color=region_colors[row["kilnregion"]],
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=320),
    ).add_to(region_groups[row["kilnregion"]])

folium.LayerControl(collapsed=True).add_to(m)

# --- Proportional-symbol size legend
legend_rows = "".join(
    f'<div style="display:flex;align-items:center;gap:8px">'
    f'<span style="display:inline-block;width:{b["radius_px"]*2}px;'
    f'height:{b["radius_px"]*2}px;border-radius:50%;background:#888;'
    f'opacity:0.55;border:1px solid #555"></span>{b["label"]}</div>'
    for b in BUCKETS
)
legend_html = (
    '<div style="position:fixed;bottom:30px;left:10px;z-index:1000;'
    'background:white;padding:8px 10px;border:1px solid #bbb;'
    'border-radius:4px;font-size:12px;line-height:1.6">'
    '<b>Potter activity (score)</b><br>'
    f'{legend_rows}</div>'
)
m.get_root().html.add_child(Element(legend_html))

m


## Step 4 — Explore

Three dataframes are in scope — `df_assignments` (raw potter × centre pairs), `df_centres` (centre geographies), and `df_activity` (the join used for the map). The example below lists the potters with the highest-score assignments to a single centre; change the filter to ask your own question.


In [ ]:
# Hier anpassen: change the centre name, explore vague (score < 1) assignments…
target = df_activity.iloc[0]["itemLabel"]
print(f"Top potters at the most-active centre: {target}\n")
(df_assignments[df_assignments["itemLabel"] == target]
    .sort_values("score", ascending=False)
    .head(10)[["actorEntityLabel", "score"]])


---

*Part of an Open Educational Resource series on knowledge graphs and linked open data, produced in the context of [NFDI4Objects](https://www.nfdi4objects.net/).*
